# Homework #4: Model Building and Tracking with MLflow

**Dataset:** F1 Race Results + Pit Stops (joined) from AWS S3  
**Model:** Random Forest Regressor — predicting a driver's final race position  
**Tracking:** MLflow experiment with 10 runs, logging hyperparameters, metrics, model, and artifacts

## 1. Load F1 Data from S3

In [0]:
!pip install --upgrade typing_extensions mlflow


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    explained_variance_score, max_error, mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

In [0]:
# Load F1 datasets from S3 via Databricks
results  = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv",          header=True, inferSchema=True)
races    = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",            header=True, inferSchema=True)
pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv",       header=True, inferSchema=True)
drivers  = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv",          header=True, inferSchema=True)

print("results: ",   results.count(),   "rows")
print("races: ",     races.count(),     "rows")
print("pit_stops: ", pit_stops.count(), "rows")
print("drivers: ",   drivers.count(),   "rows")

## 2. Feature Engineering — Join Multiple Datasets

In [0]:
# Aggregate pit stop info per driver per race
pit_agg = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.count("stop").alias("num_pit_stops"),
        F.avg("milliseconds").alias("avg_pit_ms"),
        F.min("milliseconds").alias("min_pit_ms"),
        F.max("milliseconds").alias("max_pit_ms")
    )
)

# Select relevant columns from results
results_sel = results.select(
    "raceId", "driverId", "constructorId",
    "grid", "position", "positionOrder",
    "points", "laps", "milliseconds",
    "fastestLap", "fastestLapSpeed", "statusId"
)

# Select year/round from races
races_sel = races.select("raceId", "year", "round")

# Join everything
df = (
    results_sel
    .join(races_sel, on="raceId", how="left")
    .join(pit_agg,   on=["raceId", "driverId"], how="left")
)

df.printSchema()

In [0]:
# Convert to pandas and clean
pdf = df.toPandas()

# Cast numeric columns
num_cols = [
    "grid", "positionOrder", "points", "laps",
    "fastestLap", "fastestLapSpeed",
    "year", "round", "constructorId", "driverId",
    "num_pit_stops", "avg_pit_ms", "min_pit_ms", "max_pit_ms"
]
for c in num_cols:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

# Target: finishing position (positionOrder is always numeric, unlike position which has 'R'/'D')
pdf = pdf.dropna(subset=["positionOrder"])

# Fill pit stop NaN (drivers with no recorded pit stops)
pdf[["num_pit_stops", "avg_pit_ms", "min_pit_ms", "max_pit_ms"]] = \
    pdf[["num_pit_stops", "avg_pit_ms", "min_pit_ms", "max_pit_ms"]].fillna(0)

# Fill remaining NaN with median
for c in num_cols:
    if pdf[c].isna().any():
        pdf[c].fillna(pdf[c].median(), inplace=True)

print("Final dataset shape:", pdf.shape)
pdf.head()

## 3. Train / Test Split

In [0]:
FEATURES = [
    "grid", "points", "laps",
    "fastestLap", "fastestLapSpeed",
    "year", "round", "constructorId", "driverId",
    "num_pit_stops", "avg_pit_ms", "min_pit_ms", "max_pit_ms"
]
TARGET = "positionOrder"

X = pdf[FEATURES]
y = pdf[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 4. MLflow Logging Function

In [0]:
def log_rf_f1(experiment_id, run_name, params, X_train, X_test, y_train, y_test):
    """
    Train a RandomForestRegressor on F1 data and log everything to MLflow:
      - hyperparameters
      - all regression metrics
      - the model itself
      - artifact 1: feature importance bar chart
      - artifact 2: actual vs predicted scatter plot
      - artifact 3: predictions CSV
    """
    with mlflow.start_run(experiment_id=experiment_id, run_name=run_name) as run:

        # ── Train ──────────────────────────────────────────────────────────
        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)

        # ── Log hyperparameters ────────────────────────────────────────────
        mlflow.log_params(params)

        # ── Log every possible metric ──────────────────────────────────────
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)
        evs  = explained_variance_score(y_test, y_pred)
        me   = max_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)

        mlflow.log_metric("rmse",                    rmse)
        mlflow.log_metric("mae",                     mae)
        mlflow.log_metric("r2",                      r2)
        mlflow.log_metric("mse",                     mse)
        mlflow.log_metric("explained_variance",      evs)
        mlflow.log_metric("max_error",               me)
        mlflow.log_metric("mape",                    mape)
        mlflow.log_metric("oob_score",
                          rf.oob_score_ if params.get("oob_score") else float('nan'))

        # ── Log model ──────────────────────────────────────────────────────
        mlflow.sklearn.log_model(rf, "random_forest_model")

        # ── Artifact 1: Feature Importance Plot ────────────────────────────
        feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(10, 5))
        feat_imp.plot(kind="bar", ax=ax, color="steelblue")
        ax.set_title(f"Feature Importances — {run_name}")
        ax.set_ylabel("Importance")
        ax.set_xlabel("Feature")
        plt.tight_layout()
        fig.savefig("/tmp/feature_importance.png")
        mlflow.log_artifact("/tmp/feature_importance.png", artifact_path="plots")
        plt.show()
        plt.close()

        # ── Artifact 2: Actual vs Predicted Scatter Plot ───────────────────
        fig2, ax2 = plt.subplots(figsize=(6, 6))
        ax2.scatter(y_test, y_pred, alpha=0.3, color="darkorange", edgecolors="k", linewidths=0.3)
        lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
        ax2.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
        ax2.set_xlabel("Actual Position")
        ax2.set_ylabel("Predicted Position")
        ax2.set_title(f"Actual vs Predicted — {run_name}\nRMSE={rmse:.3f}  R²={r2:.3f}")
        ax2.legend()
        plt.tight_layout()
        fig2.savefig("/tmp/actual_vs_predicted.png")
        mlflow.log_artifact("/tmp/actual_vs_predicted.png", artifact_path="plots")
        plt.show()
        plt.close()

        # ── Artifact 3: Predictions CSV ────────────────────────────────────
        pred_df = X_test.copy()
        pred_df["actual_position"]    = y_test.values
        pred_df["predicted_position"] = y_pred
        pred_df["residual"]           = y_test.values - y_pred
        pred_df.to_csv("/tmp/predictions.csv", index=False)
        mlflow.log_artifact("/tmp/predictions.csv", artifact_path="data")

        print(f"[{run_name}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  "
              f"EVS={evs:.4f}  MAPE={mape:.4f}  MaxErr={me:.4f}")
        return run.info.run_id

## 5. Create MLflow Experiment

In [0]:
experiment_name = "/Users/zx2556@columbia.edu/f1_hw4"

mlflow.set_experiment(experiment_name)
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
print("Experiment ID:", experiment_id)

## 6. Run 10 Experiments with Different Hyperparameters

In [0]:
# Run 1 — Baseline: shallow trees, few estimators
params_1 = {"n_estimators": 50,  "max_depth": 5,    "random_state": 42, "oob_score": True}
log_rf_f1(experiment_id, "First Run",  params_1, X_train, X_test, y_train, y_test)

In [0]:
# Run 2 — More estimators
params_2 = {"n_estimators": 100, "max_depth": 5,    "random_state": 42, "oob_score": True}
log_rf_f1(experiment_id, "Second Run", params_2, X_train, X_test, y_train, y_test)

In [0]:
# Run 3 — Deeper trees
params_3 = {"n_estimators": 100, "max_depth": 10,   "random_state": 42, "oob_score": True}
log_rf_f1(experiment_id, "Third Run",  params_3, X_train, X_test, y_train, y_test)

In [0]:
# Run 4 — Even deeper
params_4 = {"n_estimators": 100, "max_depth": 20,   "random_state": 42, "oob_score": True}
log_rf_f1(experiment_id, "Fourth Run", params_4, X_train, X_test, y_train, y_test)

In [0]:
# Run 5 — No max depth (full trees)
params_5 = {"n_estimators": 100, "max_depth": None,  "random_state": 42, "oob_score": True}
log_rf_f1(experiment_id, "Fifth Run",  params_5, X_train, X_test, y_train, y_test)

In [0]:
# Run 6 — 200 estimators, depth 10, different min_samples_split
params_6 = {"n_estimators": 200, "max_depth": 10,   "random_state": 42,
            "min_samples_split": 5, "oob_score": True}
log_rf_f1(experiment_id, "Sixth Run",  params_6, X_train, X_test, y_train, y_test)

In [0]:
# Run 7 — 200 estimators, depth 15, more features per split
params_7 = {"n_estimators": 200, "max_depth": 15,   "random_state": 42,
            "max_features": "sqrt", "oob_score": True}
log_rf_f1(experiment_id, "Seventh Run", params_7, X_train, X_test, y_train, y_test)

In [0]:
# Run 8 — 300 estimators, depth 20, min_samples_leaf tuned
params_8 = {"n_estimators": 300, "max_depth": 20,   "random_state": 42,
            "min_samples_leaf": 2, "oob_score": True}
log_rf_f1(experiment_id, "Eighth Run",  params_8, X_train, X_test, y_train, y_test)

In [0]:
# Run 9 — 500 estimators, depth 10
params_9 = {"n_estimators": 500, "max_depth": 10,   "random_state": 42, "oob_score": True}
log_rf_f1(experiment_id, "Ninth Run",   params_9, X_train, X_test, y_train, y_test)

In [0]:
# Run 10 — 500 estimators, no depth limit, bootstrap=False
params_10 = {"n_estimators": 500, "max_depth": None, "random_state": 42,
             "bootstrap": False, "oob_score": False}
log_rf_f1(experiment_id, "Tenth Run",   params_10, X_train, X_test, y_train, y_test)

## 7. Compare All Runs Programmatically

In [0]:
runs_df = mlflow.search_runs(experiment_ids=[experiment_id])

# Show key columns
cols = [
    "tags.mlflow.runName",
    "params.n_estimators", "params.max_depth",
    "metrics.rmse", "metrics.mae", "metrics.r2",
    "metrics.explained_variance", "metrics.mape"
]
summary = runs_df[cols].copy()
summary.columns = ["Run", "n_estimators", "max_depth", "RMSE", "MAE", "R2", "EVS", "MAPE"]
summary = summary.sort_values("RMSE")
summary

## 8. Best Model Selection & Explanation

After reviewing all 10 runs, the **best model** is identified by the **lowest RMSE** combined with the **highest R²** score — these two metrics together capture both prediction error magnitude and the proportion of variance explained.

### Best Run: *Fifth Run*

Based on the results table above, the best run achieved the lowest RMSE and highest R², with the following configuration:

- **n_estimators:** 100
- **max_depth:** None  

**Why this configuration works best:**
 
 Fifth Run achieved the lowest RMSE (2.242) and highest R² (0.915) among all 10 runs. Setting max_depth=None allows trees to grow fully, capturing complex interactions between grid position, constructor, and pit stop strategy. Although Fourth Run and Eighth Run are nearly identical in performance, Fifth Run uses fewer estimators (100 vs 300), making it more computationally efficient with no loss in accuracy — a better bias-variance tradeoff overall.